# GRPO Smoke Test — RLVR on Nemotron 3 Nano + v3 LoRA

**Goal:** validate the full GRPO training pipeline on a TINY subset (~10 problems × G=4 rollouts) before committing GPU budget to a real run. This kernel is meant to FAIL FAST if any compatibility issue exists.

**Strategy:**
1. Load base Nemotron 3 Nano + apply our v3 LoRA adapter (LB 0.58) as starting point.
2. Set up TRL's `GRPOTrainer` with our custom reward function (`boxed_match_reward` from `scripts/rl_reward.py`).
3. Run 10 problems × G=4 rollouts × 1 epoch = ~40 generations + 1 gradient step.
4. Verify: (a) NemotronH + TRL compatibility, (b) memory fits on 96GB RTX Pro 6000, (c) reward function fires, (d) gradient step completes without OOM.
5. If smoke test passes → scale up to real GRPO run (kernel `11-grpo-full`).

**Expected unknowns this will resolve:**
- Does vLLM work with NemotronH's Mamba-Transformer hybrid? (might need standard `generate()` fallback)
- Does the reference policy (β=0 default) really skip the reference model copy and save memory?
- What's the actual per-step wall time so we can plan a real run within budget?

**Budget:** 10 problems × ~5 min/problem-with-rollouts = ~50 min wall time expected (most of which is model load + setup). If this exceeds 2 hr we'll abort.

In [ ]:
# Cell 0: Install deps into a writable location and prepend to sys.path.
# Why --target: Kaggle's offline-deps utility scripts mount read-only at
# /kaggle/usr/lib/..., and the default pip install path tries to write .pth
# files there (fails with Errno 30 Read-only file system). --target bypasses
# the system install location entirely.
import subprocess, sys, os

EXTRA_PKGS = "/tmp/extra_pkgs"
os.makedirs(EXTRA_PKGS, exist_ok=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--target", EXTRA_PKGS,
     "trl>=0.18", "peft>=0.14", "accelerate>=1.2", "datasets",
     "nvidia-cutlass-dsl"],
    check=True,
)
sys.path.insert(0, EXTRA_PKGS)

import trl, peft, accelerate, datasets
print(f"trl={trl.__version__}, peft={peft.__version__}, accelerate={accelerate.__version__}, datasets={datasets.__version__}")
try:
    import cutlass
    print(f"cutlass available at: {cutlass.__file__}")
except ImportError as e:
    print(f"⚠️ cutlass still missing: {e}")

In [ ]:
# Cell 1: Setup — sitedir discovery, ptxas-blackwell workaround, seeds
import os, random, shutil, site, stat, sys, time, glob, json
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f'[{time.perf_counter() - T0:7.1f}s] {label}', flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Find python_packages from attached utility scripts (mamba_ssm, etc.)
for utility_root in ('/kaggle/input/notebooks', '/kaggle/usr/lib/notebooks', '/kaggle/lib'):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if 'models' in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == 'python_packages':
            site.addsitedir(root)

# ptxas-blackwell exec workaround (Triton needs an executable copy)
PTXAS_DST = '/tmp/ptxas-blackwell'
for src in (
    '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell',
    '/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell',
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_PATH'] = PTXAS_DST
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = PTXAS_DST
        break

# Try to import TRL — fail fast if not available in Kaggle env or offline-deps
try:
    import trl
    step(f'TRL version: {trl.__version__}')
except ImportError as e:
    print(f'⚠️ TRL not available: {e}')
    print('Need to add trl to offline-dependencies or pre-install in Kaggle env.')
    raise
step(f'Setup done; seeds set to {SEED}')

In [ ]:
# Cell 2: Inline the reward function (kernel can't import from scripts/ in our repo)
import re

BOXED_RE = re.compile(r'\\boxed\{([^}]*)\}')

def _extract_boxed(c):
    ms = list(BOXED_RE.finditer(c))
    return ms[-1].group(1).strip() if ms else ''

def _numerically_equal(a, b):
    try:
        return float(a) == float(b)
    except (ValueError, TypeError):
        return False

def _gold_in_completion(completion, gold):
    pattern = r'(?<!\d)' + re.escape(gold) + r'(?!\d)'
    return bool(re.search(pattern, completion))

def score_completion(completion, gold):
    if not gold: return 0.0
    gold = gold.strip()
    boxed = _extract_boxed(completion)
    if boxed and boxed == gold:
        return 1.0
    if boxed and _numerically_equal(boxed, gold):
        return 0.3
    if _gold_in_completion(completion, gold):
        return 0.1
    return 0.0

def boxed_match_reward(completions, gold_answer=None, **kwargs):
    if gold_answer is None:
        raise ValueError("Dataset must have 'gold_answer' column.")
    return [score_completion(c, g) for c, g in zip(completions, gold_answer)]

# Quick self-test
assert score_completion('...\\boxed{57.0}<|im_end|>', '57.0') == 1.0
assert score_completion('...\\boxed{57.00}<|im_end|>', '57.0') == 0.3
assert score_completion('...\\boxed{57.03}<|im_end|>', '57.0') == 0.0
step('Reward function loaded and self-test passed')

In [ ]:
# Cell 3: Load base model + tokenizer + apply v3 LoRA adapter as starting point
import kagglehub
import mamba_ssm  # noqa: F401
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
step('Tokenizer loaded')

# Load v3 adapter (LB 0.58) as our starting point — GRPO will fine-tune from here
ADAPTER_CANDIDATES = [
    '/kaggle/input/datasets/gideonnazarian/nemotron-lora-sft-v3-adapter',
    '/kaggle/input/nemotron-lora-sft-v3-adapter',
]
ADAPTER_PATH = None
for p in ADAPTER_CANDIDATES:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, 'adapter_config.json')):
        ADAPTER_PATH = p
        break
if ADAPTER_PATH is None:
    raise FileNotFoundError(f'v3 adapter not found in {ADAPTER_CANDIDATES}')
step(f'v3 adapter at {ADAPTER_PATH}')

t = time.perf_counter()
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map='auto', trust_remote_code=True, dtype=torch.bfloat16,
)
step(f'Base model loaded in {time.perf_counter() - t:.0f}s, VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB')

# Attach v3 adapter and make it trainable (GRPO will update these weights)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()
step(f'v3 adapter attached + trainable. VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Cell 4: Load tiny RL dataset for smoke test (10 problems mixed across categories)
from datasets import Dataset

RL_DATA_CANDIDATES = [
    '/kaggle/input/datasets/gideonnazarian/nemotron-solver-training-data/rl-training-data.jsonl',
    '/kaggle/input/nemotron-solver-training-data/rl-training-data.jsonl',
]
RL_DATA = None
for p in RL_DATA_CANDIDATES:
    if os.path.exists(p):
        RL_DATA = p
        break
if RL_DATA is None:
    hits = glob.glob('/kaggle/input/**/rl-training-data.jsonl', recursive=True)
    if hits:
        RL_DATA = hits[0]
if RL_DATA is None:
    raise FileNotFoundError(f'rl-training-data.jsonl not found. /kaggle/input/ contains: {os.listdir("/kaggle/input")}')
step(f'RL data at {RL_DATA}')

all_records = []
with open(RL_DATA) as f:
    for line in f:
        all_records.append(json.loads(line))
step(f'Loaded {len(all_records)} total RL records')

# Pick 2 problems per category for the smoke test (stratified, deterministic)
SMOKE_PER_CAT = 2
by_cat = {}
for r in all_records:
    by_cat.setdefault(r['category'], []).append(r)

rng = random.Random(SEED)
smoke = []
for cat, recs in sorted(by_cat.items()):
    smoke.extend(rng.sample(recs, min(SMOKE_PER_CAT, len(recs))))
step(f'Smoke-test sample: {len(smoke)} records across {len(by_cat)} categories')

# Strip down to just (prompt, gold_answer) — TRL passes other columns to reward fn as kwargs
ds = Dataset.from_list([
    {'prompt': r['prompt'], 'gold_answer': r['gold_answer']}
    for r in smoke
])
print('Smoke dataset:', ds)
print('Example record:', ds[0])

In [ ]:
# Cell 5: Set up GRPOTrainer and run a SINGLE step
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='/kaggle/working/grpo_smoke',
    per_device_train_batch_size=1,       # one prompt per step (memory-conservative)
    num_generations=4,                    # G=4 rollouts per prompt (GRPO group size)
    max_completion_length=512,            # smoke test: short completions for speed
    max_prompt_length=1024,
    num_train_epochs=1,
    learning_rate=5e-6,                   # lower than SFT — RL is sensitive
    beta=0.0,                             # no KL term → no reference model needed (saves ~60GB)
    scale_rewards=True,
    bf16=True,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    seed=SEED,
    # vLLM: try enabled first, fall back if NemotronH incompatible
    use_vllm=False,                       # START with HF generate(); enable vLLM later if smoke passes
)
step('GRPOConfig built')

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=boxed_match_reward,
    train_dataset=ds,
    args=config,
)
step(f'GRPOTrainer built. VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB')

step('Starting GRPO training (smoke test)')
trainer.train()
step('Training done')

In [ ]:
# Cell 6: Save smoke-test adapter (for inspection only — not for submission)
OUTPUT_DIR = '/kaggle/working'
for f in os.listdir(OUTPUT_DIR):
    if f.startswith('adapter') or f.endswith('.safetensors') or f.endswith('.bin'):
        try: os.remove(os.path.join(OUTPUT_DIR, f))
        except (IsADirectoryError, PermissionError): pass

model.save_pretrained(OUTPUT_DIR)
step(f'Adapter saved to {OUTPUT_DIR}')
import subprocess
subprocess.run('ls -la /kaggle/working/', shell=True, check=False)
step(f'Total wall time: {(time.perf_counter() - T0) / 60:.1f} min')
print('Smoke test done. If this completed without OOM and reward metrics look sensible, scale up to a full GRPO run.')